In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

In [2]:

orders = pd.read_csv(r'olist_orders_dataset.csv')
customers = pd.read_csv(r'olist_customers_dataset.csv')
order_items = pd.read_csv(r'olist_order_items_dataset.csv')
products = pd.read_csv(r'olist_products_dataset.csv')
payments = pd.read_csv(r'olist_order_payments_dataset.csv')
reviews = pd.read_csv(r'olist_order_reviews_dataset.csv')
sellers = pd.read_csv(r'olist_sellers_dataset.csv')
category_translation = pd.read_csv(r'product_category_name_translation.csv')

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
datasets = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'products': products,
    'payments': payments,
    'reviews': reviews,
    'sellers': sellers
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name.upper()} — Shape: {df.shape}")
    print(f"Nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")


ORDERS — Shape: (99441, 8)
Nulls:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

CUSTOMERS — Shape: (99441, 5)
Nulls:
Series([], dtype: int64)

ORDER_ITEMS — Shape: (112650, 7)
Nulls:
Series([], dtype: int64)

PRODUCTS — Shape: (32951, 9)
Nulls:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

PAYMENTS — Shape: (103886, 5)
Nulls:
Series([], dtype: int64)

REVIEWS — Shape: (99224, 7)
Nulls:
review_comment_title      87656
review_comment_message    58247
dtype: int64

SELLERS — Shape: (3095, 4)
Nulls:
Series([], dtype: int64)


In [4]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Datetime columns fixed!")

Datetime columns fixed!


In [5]:
products = products.merge(category_translation, on='product_category_name', how='left')
products['product_category_name_english'].fillna('unknown', inplace=True)
print("Categories translated!")

Categories translated!


C:\Users\shirl\AppData\Local\Temp\ipykernel_31672\43972869.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  products['product_category_name_english'].fillna('unknown', inplace=True)


In [6]:
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(order_items, on='order_id', how='left')
df = df.merge(products[['product_id','product_category_name_english']], on='product_id', how='left')
df = df.merge(payments[['order_id','payment_type','payment_value']], on='order_id', how='left')
df = df.merge(reviews[['order_id','review_score']], on='order_id', how='left')

print(f"Master dataframe shape: {df.shape}")
df.head()

Master dataframe shape: (119143, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name_english,payment_type,payment_value,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,housewares,credit_card,18.12,4.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,housewares,voucher,2.00,4.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,housewares,voucher,18.59,4.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumery,boleto,141.46,4.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,auto,credit_card,179.12,5.0


In [7]:
# Keep only delivered orders
df = df[df['order_status'] == 'delivered']

# Drop rows where price is null
df.dropna(subset=['price'], inplace=True)

# Fill missing review scores with median
df['review_score'].fillna(df['review_score'].median(), inplace=True)

# Add delivery delay column (positive = late, negative = early)
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

# Add month and year columns
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['order_year'] = df['order_purchase_timestamp'].dt.year

print(f"Clean dataframe shape: {df.shape}")
print(f"Null values remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

C:\Users\shirl\AppData\Local\Temp\ipykernel_31672\3548138799.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['review_score'].fillna(df['review_score'].median(), inplace=True)


Clean dataframe shape: (115723, 25)
Null values remaining:
order_approved_at                15
order_delivered_carrier_date      2
order_delivered_customer_date     8
payment_type                      3
payment_value                     3
delivery_delay_days               8
dtype: int64


In [8]:
df.to_csv(r'C:\Users\shirl\OneDrive\Desktop\ecommerce-analytics\data\clean_orders.csv', index=False)
print("Clean dataset saved")

Clean dataset saved
